# Spark-NLP


## Lemma 
Create PySpark environment and import library

In [1]:
# Import Spark NLP
import sparknlp
from sparknlp.base import *
from sparknlp.annotator import *
from pyspark.ml import Pipeline

In [2]:
# In this example, the lemma dictionary `lemmas_small.txt` has the form of
#
# ...
# pick	->	pick	picks	picking	picked
# peck	->	peck	pecking	pecked	pecks
# pickle	->	pickle	pickles	pickled	pickling
# pepper	->	pepper	peppers	peppered	peppering
# ...
#
# where each key is delimited by `->` and values are delimited by `\t`

spark = sparknlp.start()


documentAssembler = DocumentAssembler().setInputCol("value").setOutputCol("document")

sentenceDetector = SentenceDetector().setInputCols(["document"]).setOutputCol("sentence")

tokenizer = Tokenizer().setInputCols(["sentence"]).setOutputCol("token")

lemmatizer = Lemmatizer().setInputCols(["token"]).setOutputCol("lemma").setDictionary("data/lemmas_small.txt", "->", "\t")

pipeline = Pipeline().setStages([documentAssembler,sentenceDetector,tokenizer,lemmatizer])

data = spark.createDataFrame([["Peter Pipers employees are picking pecks of pickled peppers."]]).toDF("value")

result = pipeline.fit(data).transform(data)
result.selectExpr("lemma.result").show(truncate=False)

+------------------------------------------------------------------+
|result                                                            |
+------------------------------------------------------------------+
|[Peter, Pipers, employees, are, pick, peck, of, pickle, pepper, .]|
+------------------------------------------------------------------+



In [3]:
print(result)

DataFrame[value: string, document: array<struct<annotatorType:string,begin:int,end:int,result:string,metadata:map<string,string>,embeddings:array<float>>>, sentence: array<struct<annotatorType:string,begin:int,end:int,result:string,metadata:map<string,string>,embeddings:array<float>>>, token: array<struct<annotatorType:string,begin:int,end:int,result:string,metadata:map<string,string>,embeddings:array<float>>>, lemma: array<struct<annotatorType:string,begin:int,end:int,result:string,metadata:map<string,string>,embeddings:array<float>>>]


In [4]:
df_text = spark.read.text("data/2.txt")
df_text.printSchema()
df_text.show(truncate=False)

root
 |-- value: string (nullable = true)

+-----------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                  |
+-----------------------------------------------------------------------------------------------------------------------+
|Lorem Ipsum is simply dummy text of the printing                                                                       |
|The standard chunk of Lorem Ipsum used since the 1500s                                                                 |
|There are many variations of passages of Lorem Ipsum available, but the majority have suffered alteration in some form.|
|Lorem ipsum dolor sit amet, consectetur adipiscing elit.                                                               |
|Mauris ut dapibus lacus. Maecenas sollicitudin ex et condimentum suscipit.                            

In [5]:
type(df_text)

pyspark.sql.dataframe.DataFrame

In [6]:
result = pipeline.fit(df_text).transform(df_text)
rdd = result.selectExpr("lemma.result").rdd
df=result.selectExpr("lemma.result")

#rdd = rdd.map(lambda x: x.toString()).toDF()  # has error

print(rdd.collect())

print(df.collect())

#rdd.saveAsTextFile("data/lemma_test")


[Row(result=['Lorem', 'Ipsum', 'is', 'simply', 'dummy', 'text', 'of', 'the', 'print']), Row(result=['The', 'standard', 'chunk', 'of', 'Lorem', 'Ipsum', 'used', 'since', 'the', '1500s']), Row(result=['There', 'are', 'many', 'variations', 'of', 'passage', 'of', 'Lorem', 'Ipsum', 'available', ',', 'but', 'the', 'majority', 'have', 'suffered', 'alteration', 'in', 'some', 'form', '.']), Row(result=['Lorem', 'ipsum', 'dolor', 'sit', 'amet', ',', 'consectetur', 'adipiscing', 'elit', '.']), Row(result=['Mauris', 'ut', 'dapibus', 'lacus', '.', 'Maecenas', 'sollicitudin', 'ex', 'et', 'condimentum', 'suscipit', '.']), Row(result=['Lorem', 'Ipsum', 'is', 'simply', 'dummy', 'text', 'of', 'the', 'print', 'and', 'typesetting', 'industry', '.']), Row(result=['Contrary', 'to', 'popular', 'belief', ',', 'Lorem', 'Ipsum', 'is', 'not', 'simply', 'random', 'text', '.']), Row(result=['The', 'standard', 'chunk', 'of', 'Lorem', 'Ipsum', 'used', 'since', 'the', '1500s', 'is', 'reproduced', 'below', 'for', 'tho

In [7]:
outputFile = "data/lemma_test2"
#open(outputFile, 'a+', encoding = 'utf-8') as targetFile
df = result.selectExpr("lemma.result")
print(df.collect())
#df.write.text(outputFile)   # has error

[Row(result=['Lorem', 'Ipsum', 'is', 'simply', 'dummy', 'text', 'of', 'the', 'print']), Row(result=['The', 'standard', 'chunk', 'of', 'Lorem', 'Ipsum', 'used', 'since', 'the', '1500s']), Row(result=['There', 'are', 'many', 'variations', 'of', 'passage', 'of', 'Lorem', 'Ipsum', 'available', ',', 'but', 'the', 'majority', 'have', 'suffered', 'alteration', 'in', 'some', 'form', '.']), Row(result=['Lorem', 'ipsum', 'dolor', 'sit', 'amet', ',', 'consectetur', 'adipiscing', 'elit', '.']), Row(result=['Mauris', 'ut', 'dapibus', 'lacus', '.', 'Maecenas', 'sollicitudin', 'ex', 'et', 'condimentum', 'suscipit', '.']), Row(result=['Lorem', 'Ipsum', 'is', 'simply', 'dummy', 'text', 'of', 'the', 'print', 'and', 'typesetting', 'industry', '.']), Row(result=['Contrary', 'to', 'popular', 'belief', ',', 'Lorem', 'Ipsum', 'is', 'not', 'simply', 'random', 'text', '.']), Row(result=['The', 'standard', 'chunk', 'of', 'Lorem', 'Ipsum', 'used', 'since', 'the', '1500s', 'is', 'reproduced', 'below', 'for', 'tho

In [8]:
data = spark.createDataFrame([["car cars car's cars' "]]).toDF("value")

result = pipeline.fit(data).transform(data)
result.selectExpr("lemma.result").show(truncate=False)

+-----------------------+
|result                 |
+-----------------------+
|[car, car, car, car, ']|
+-----------------------+



In [9]:
spark.stop()